# CosmosDB Live Container Migration Validation (Microsoft Fabric)

Parameterized validation notebook for multi-container migrations. Compares source and target
containers to identify documents not yet migrated.

In [7]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Cosmos_Migration"
    }
}

StatementMeta(, 27d83d5f-a1ba-4a49-ae74-9c8685901734, -1, Finished, Available, Finished, True)

In [ ]:
# Parameters

import uuid

# Source config
cosmosSourceEndpoint = "https://sourcedata.documents.azure.com:443/"  # Cosmos DB Account URI of the source account
cosmosSourceMasterKey = ""  # Cosmos DB Account PRIMARY KEY of the source account
cosmosSourceDatabaseName = "SampleDB"  # name of your source database
cosmosSourceContainerName = "SampleContainer"  # name of the container you want to validate

# Target config
cosmosTargetEndpoint = "https://targetdata.documents.azure.com:443/"  # Cosmos DB Account URI of the target account
cosmosTargetMasterKey = ""  # Cosmos DB Account PRIMARY KEY of the target account
cosmosTargetDatabaseName = "SampleDB"  # name of your target database
cosmosTargetContainerName = "CopiedContainer1"  # name of the target container

# Common config
cosmosRegion = "[West US 2]"  # Cosmos DB Region

In [12]:
# Helper functions — uses DataFrame API directly (no shared Spark SQL catalogs)

from pyspark.sql.functions import col


def read_cosmos_container(endpoint, key, database, container, infer_schema="true"):
    """Read a Cosmos DB container directly via the DataFrame API."""
    return (spark.read
        .format("cosmos.oltp")
        .option("spark.cosmos.accountEndpoint", endpoint)
        .option("spark.cosmos.accountKey", key)
        .option("spark.cosmos.database", database)
        .option("spark.cosmos.container", container)
        .option("spark.cosmos.read.inferSchema.enabled", infer_schema)
        .option("spark.cosmos.read.inferSchema.includeSystemProperties", "true")
        .option("spark.cosmos.read.partitioning.strategy", "Restrictive")
        .load()
    )

StatementMeta(, 27d83d5f-a1ba-4a49-ae74-9c8685901734, 9, Finished, Available, Finished, False)

In [ ]:
# Read source and target containers

print(f"Reading SOURCE: {cosmosSourceEndpoint} / {cosmosSourceDatabaseName} / {cosmosSourceContainerName}")
source_df = (read_cosmos_container(
        cosmosSourceEndpoint, cosmosSourceMasterKey,
        cosmosSourceDatabaseName, cosmosSourceContainerName,
        infer_schema="true")
    .select("id", "_etag")
)

print(f"Reading TARGET: {cosmosTargetEndpoint} / {cosmosTargetDatabaseName} / {cosmosTargetContainerName}")
target_df = (read_cosmos_container(
        cosmosTargetEndpoint, cosmosTargetMasterKey,
        cosmosTargetDatabaseName, cosmosTargetContainerName,
        infer_schema="true")
    .select("id", "_origin_etag")
)

In [ ]:
# Compare source and target — documents in source but not (or changed) in target

sourceTargetDiff = source_df.alias("source").join(
    target_df.alias("target"),
    (col("source.id") == col("target.id")) & (col("source._etag") == col("target._origin_etag")),
    "left_anti"
)

sourceTargetCountDiff = sourceTargetDiff.count()
print(f"Documents in source not matched in target: {sourceTargetCountDiff}")

In [ ]:
# This shouldn't return any results if migration is complete;
# otherwise the missing records will be displayed

display(sourceTargetDiff)

In [ ]:
# Count of rows in source but not in target

display(sourceTargetCountDiff)